# Chapter 5: Eager and Lazy APIs

In [1]:
import polars as pl
pl.__version__  # The book is built with Polars version 1.20.0

'1.33.0'

## 5.1 Eager API: DataFrame

In [ ]:
%%time
trips = pl.read_parquet("data/taxi/yellow_tripdata_*.parquet")  
sum_per_vendor = trips.group_by("VendorID").sum()  

income_per_distance_per_vendor = sum_per_vendor.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

top_three = income_per_distance_per_vendor.sort(  
    by="income_per_distance", descending=True
).head(3)

top_three

CPU times: total: 24.8 s
Wall time: 2.61 s


VendorID,income_per_distance
i64,f64
1,6.434789
6,5.296493
5,4.731557


## 5.2 Lazy API: LazyFrame

LazyFrame pre-audit operations, schema error exposed early.

In [ ]:
# This raises a SchemaError:
# names_lf = pl.LazyFrame({"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35]})

# erroneous_query = names_lf.with_columns(
#     sliced_age=pl.col("age").str.slice(1, 3)
# )

# result_df = erroneous_query.collect()

## 5.3 Performance Differences

`read_parquet()` is eager, `scan_parquet()` is lazy

In [ ]:
%%time
trips = pl.scan_parquet("data/taxi/yellow_tripdata_*.parquet")
sum_per_vendor = trips.group_by("VendorID").sum()

income_per_distance_per_vendor = sum_per_vendor.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

top_three = income_per_distance_per_vendor.sort(
    by="income_per_distance", descending=True
).head(3)

top_three.collect()

CPU times: total: 3.61 s
Wall time: 460 ms


VendorID,income_per_distance
i64,f64
1,6.434789
6,5.296493
5,4.731557


In [9]:
lf = pl.LazyFrame({"col1": [1, 2, 3], "col2": [4, 5, 6]})

# ... Some heavy computation ...

print(lf.collect())

print(lf.with_columns(pl.col("col1") + 1).collect())  

shape: (3, 2)
┌──────┬──────┐
│ col1 ┆ col2 │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 1    ┆ 4    │
│ 2    ┆ 5    │
│ 3    ┆ 6    │
└──────┴──────┘
shape: (3, 2)
┌──────┬──────┐
│ col1 ┆ col2 │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 2    ┆ 4    │
│ 3    ┆ 5    │
│ 4    ┆ 6    │
└──────┴──────┘


## 5.4 Functionality Differences

LazyFrame data is NOT accessible before `lf.collect()`

### 5.4.1 Attributes

DataFrame has columns, dtypes, flags, height, width, schema, shape<br>
LazyFrame doesn't have flags, height, width

### 5.4.2 Aggregation Methods

LF doesn't support `xxx_horizontal()` methods, because not knowing cross-col data structure

### 5.4.3 Computation Methods

`df.fold()` combines 2 cols into 1<br>
`df.hash_rows()` hash rows into `UInt64` value<br>
LF NOT support

### 5.4.4 Descriptive Methods

DF supports describe, estimated_size, glimpse, is_duplicated, is_unique, null_count...<br>
LF supports explain, null_count, show_graph

### 5.4.5 GroupBy Methods

DF and LF support all aggregate func. DF can iterate.

### 5.4.6 Exporting Methods

LF has no data, thus can't export. DF and LF can .serialize()

### 5.4.7 Manipulation and Selection Methods

### 5.4.8 Miscellaneous Methods

## 5.5 Tips and Tricks

### 5.5.1 Going from LazyFrame to DataFrame and Vice Versa

DF.lazy() -> LF, LF.collect() -> DF

### 5.5.2 Joining a DataFrame with a LazyFrame

DF and LF can't join directly

In [ ]:
# This raises a TypeError:
# big_sales_data = pl.LazyFrame(
#     {"sale_id": [101, 102, 103], "amount": [250, 150, 300]}
# )
#
# sales_metadata = pl.DataFrame(
#     {"sale_id": [101, 102, 103], "category": ["A", "B", "A"]}
# )
#
# big_sales_data.join(sales_metadata, on="sale_id").collect()

Switch to the same frame type and join

In [10]:
big_sales_data = pl.LazyFrame(
    {"sale_id": [101, 102, 103], "amount": [250, 150, 300]}
)

sales_metadata = pl.DataFrame(
    {"sale_id": [101, 102, 103], "category": ["A", "B", "A"]}
)

big_sales_data.join(sales_metadata.lazy(), on="sale_id").collect()

sale_id,amount,category
i64,i64,str
101,250,"""A"""
102,150,"""B"""
103,300,"""A"""


### 5.5.3 Caching Intermittent Results

Use `LF.collect().lazy()` to cache result of complex calculation<br>
result now in memory

In [11]:
lf = pl.LazyFrame({"col1": [1, 2, 3], "col2": [4, 5, 6]})

# ... Some heavy computation ...

lf = lf.collect().lazy()  
print(lf.collect())

print(lf.with_columns(pl.col("col1") + 1).collect())  

shape: (3, 2)
┌──────┬──────┐
│ col1 ┆ col2 │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 1    ┆ 4    │
│ 2    ┆ 5    │
│ 3    ┆ 6    │
└──────┴──────┘
shape: (3, 2)
┌──────┬──────┐
│ col1 ┆ col2 │
│ ---  ┆ ---  │
│ i64  ┆ i64  │
╞══════╪══════╡
│ 2    ┆ 4    │
│ 3    ┆ 5    │
│ 4    ┆ 6    │
└──────┴──────┘


## Takeaways